# Alignment parameter sweep after note2

Continue the completed cached-pitch note-detection experiment, holding its selected segmentation and score timing fixed. By default, use **note2's highest-F1 candidate with no added clean false alarms**; set `NOTE_VARIANT` to a saved variant ID to choose another setting. The selected settings are displayed before execution.

Sweep pitch/time weights, onset/duration balance, and independent insertion/deletion costs. Always include the saved alignment baseline and a controlled **pitch weight 2 versus 4** comparison. No production defaults or source caches are modified.

This is exploratory tuning on the same injected CocoChorales sample, **not held-out validation**. Repeat correction is excluded to isolate alignment costs, matching note2. Validate finalists on new chorales and with production repeat correction before promotion.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (p / "app.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from benchmarks.modules.mistake.sweeps.AlignmentParamSweep import Axes, PARAMS, grid, load_source, run_sweep

NOTE_RUN = REPO_ROOT / "benchmarks/results/sweeps/note2"
OUTPUT = REPO_ROOT / "benchmarks/results/sweeps/mistake2"
NOTE_VARIANT = None  # None: saved clean-safe leader; or an explicit note2 variant ID.
WORKERS = max(1, os.cpu_count() or 1)
RESUME = True
AXES = Axes(
    alignment_gamma_pitch=(1., 2., 3., 4., 6.),
    alignment_gamma_time=(.5, 1., 2., 4.),
    alignment_alpha_onset=(0., .25, .5, 1.),
    ins_cost=(3., 5., 7.),
    del_cost=(3., 5., 7.),
)


## Inspect the fixed notes and alignment grid

For paired notes, cost is `gamma_pitch × pitch_error + gamma_time × (alpha_onset × onset_error + alpha_duration × duration_error)`. Pitch error is in semitones; temporal errors are in seconds. `alpha_duration = 1 - alpha_onset`. Unmatched notes cost `ins_cost + gamma_time × user_duration` or `del_cost + gamma_time × score_duration`.

The default Cartesian grid contains 720 settings. Baseline and pitch-only 2/4 controls are added if absent. Pitch tolerance, truth events, pitches, instrument ranges, voicing gates, note boundaries, and fitted score timing stay fixed. Clean seeds are deduplicated exactly as in note2.


In [ ]:
cases, fixed_notes, baseline, selected_note_variant = load_source(NOTE_RUN, NOTE_VARIANT)
variants = grid(baseline, AXES)
print("Selected note2 variant:", selected_note_variant)
print("Fixed note parameters:", fixed_notes)
print("Saved alignment baseline:", baseline)
sample = pd.DataFrame([{k: c[k] for k in ("case_id", "instrument", "seed", "rate")} for c in cases])
display(sample.groupby(["instrument", "rate"]).size().unstack(fill_value=0))
print(f"{len(cases)} cases × {len(variants)} settings = {len(cases) * len(variants):,} evaluations")
print("Output:", OUTPUT)


## Run or resume

Spawned workers process cases in parallel with one numerical-library thread each. A worker segments its case once, reproduces the selected note2 variant's saved TP/FP/FN at every tolerance, then changes only alignment costs. Any baseline mismatch fails the run. Each completed case is checkpointed using input, code, package, grid, and selected-note hashes. Interruptions retain finished case checkpoints.

Rank by pooled mistake-event F1 at **100 ms / 50 cents**, subject to no increase in clean false alarms. These are missed/extra event metrics; substitutions can contribute both events. Timing-mistake labels are not the objective. Report 50/200 ms sensitivity as well. F1 is calculated from pooled TP/FP/FN, not averaged per-case F1.


In [ ]:
rows, summary = run_sweep(NOTE_RUN, OUTPUT, note_variant=selected_note_variant,
                          axes=AXES, workers=WORKERS, resume=RESUME)
metadata = json.loads((OUTPUT / "run.json").read_text())
print(f"Finished {metadata['evaluations']:,} evaluations in {metadata['wall_seconds']:.1f}s")


## Reload saved results and inspect leaders

This cell can run after setup without rerunning the sweep. `clean_safe` is relative to the saved alignment baseline on the **selected note settings**, not the original pre-note2 segmentation. Tied rows are sorted by variant ID, **not preference or statistical superiority**. The leading row may show pitch weight 6 even when 3 and 4 give identical results. The cell below counts tied leaders and explicitly prefers the prespecified pitch-weight-4-only change when it ties for best; the controlled table shows whether 6 adds any benefit.


In [ ]:
metadata = json.loads((OUTPUT / "run.json").read_text())
if metadata["status"] != "complete":
    raise RuntimeError("Finish the sweep before interpreting its results")
rows = pd.read_csv(OUTPUT / "rows.csv")
summary = pd.read_csv(OUTPUT / "summary.csv")
baseline = metadata["baseline"]
print("Results use note variant:", metadata["note_variant"], metadata["fixed_notes"])
COLUMNS = PARAMS + ["injected_audio_pitch100_f1", "injected_precision", "injected_recall",
                   "mistake_f1_gain_pp", "clean_audio_pitch100_fp", "clean_false_alarms_per_100_notes", "clean_safe"]
print("Baseline with fixed note2 segmentation")
display(summary.loc[summary.baseline, COLUMNS].round(3))
print("Top 20, clean-control constraint first")
display(summary[COLUMNS].head(20).round(3))
print("Unconstrained F1 leader (diagnostic)")
display(summary.sort_values("injected_audio_pitch100_f1", ascending=False)[COLUMNS].head(1).round(3))
leaders = summary[summary.clean_safe & np.isclose(
    summary.injected_audio_pitch100_f1,
    summary.loc[summary.clean_safe, "injected_audio_pitch100_f1"].max())]
print(f"{len(leaders)} configurations tie for the best clean-safe F1; the first row is not a unique winner.")
# Prefer the prespecified one-knob experiment when it ties for best.
# This is a simplicity/prior-hypothesis choice, not evidence that 4 beats 3 or 6.
preferred_params = dict(baseline, alignment_gamma_pitch=4.)
preferred = leaders[np.logical_and.reduce([
    np.isclose(leaders[p], preferred_params[p]) for p in PARAMS])]
best = preferred.iloc[0] if len(preferred) else leaders.iloc[0]
print("Candidate for promotion (one-knob pitch-4 control when tied):", {p: best[p] for p in PARAMS})
print("Controlled pitch-weight comparison, with every other cost at baseline:")
controls = summary[np.logical_and.reduce([
    np.isclose(summary[p], baseline[p]) for p in PARAMS if p != "alignment_gamma_pitch"])]
display(controls.sort_values("alignment_gamma_pitch")[PARAMS + [
    "injected_audio_pitch100_tp", "injected_audio_pitch100_fp", "injected_audio_pitch100_fn",
    "injected_audio_pitch100_f1", "clean_audio_pitch100_fp"]].round(3))


## Isolate pitch weight 2 → 4

Hold every other alignment parameter at its saved value. This isolates the earlier pitch-weight hypothesis from improvements due to combinations of cost changes. The heatmap then shows pitch/time combinations at baseline onset/duration balance and gap costs.


In [ ]:
other_params = [p for p in PARAMS if p != "alignment_gamma_pitch"]
pitch_only = summary[np.logical_and.reduce([np.isclose(summary[p], baseline[p]) for p in other_params])]
display(pitch_only.sort_values("alignment_gamma_pitch")[COLUMNS].round(3))
p2 = pitch_only[np.isclose(pitch_only.alignment_gamma_pitch, 2.)].iloc[0]
p4 = pitch_only[np.isclose(pitch_only.alignment_gamma_pitch, 4.)].iloc[0]
print(f"Pitch 2 → 4: {p4.injected_audio_pitch100_f1 - p2.injected_audio_pitch100_f1:+.3f} F1 percentage points; "
      f"{int(p4.clean_audio_pitch100_fp - p2.clean_audio_pitch100_fp):+d} clean false alarms")
held = [p for p in PARAMS if p not in ("alignment_gamma_pitch", "alignment_gamma_time")]
surface = summary[np.logical_and.reduce([np.isclose(summary[p], baseline[p]) for p in held])]
heatmap = surface.pivot(index="alignment_gamma_pitch", columns="alignment_gamma_time", values="injected_audio_pitch100_f1").sort_index()
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap, aspect="auto", origin="lower", cmap="viridis")
ax.set_xticks(range(len(heatmap.columns)), labels=heatmap.columns)
ax.set_yticks(range(len(heatmap.index)), labels=heatmap.index)
for i in range(len(heatmap.index)):
    for j in range(len(heatmap.columns)):
        if pd.notna(heatmap.iloc[i, j]):
            ax.text(j, i, f"{heatmap.iloc[i, j]:.1f}", ha="center", va="center", color="white")
ax.set(xlabel="Time weight", ylabel="Pitch weight", title="Mistake F1 with baseline balance and gap costs")
fig.colorbar(im, ax=ax, label="Pooled mistake F1 (%)")
fig.tight_layout()
plt.show()


## Sensitivity and paired case breakdown

Compare the selected alignment candidate with its baseline on each instrument and injection seed. Both seeds belong to the tuning sample. Clean cases are reported separately; their F1 is undefined when there are no truth events and no false alarms.


In [ ]:
selected = summary[summary.baseline | summary.variant.eq(best.variant)]
display(selected[PARAMS + ["injected_audio_pitch50_f1", "injected_audio_pitch100_f1",
                          "injected_audio_pitch200_f1", "injected_audio_extra100_f1",
                          "injected_audio_missed100_f1", "clean_audio_pitch100_fp"]].round(3))
comparison = rows[rows.baseline | rows.variant.eq(best.variant)].copy()
comparison["setting"] = np.where(comparison.baseline, "baseline", "selected")
for keys in (["instrument", "setting"], ["seed", "setting"]):
    totals = comparison[comparison.rate > 0].groupby(keys)[["audio_pitch100_tp", "audio_pitch100_fp", "audio_pitch100_fn"]].sum()
    denominator = 2 * totals.audio_pitch100_tp + totals.audio_pitch100_fp + totals.audio_pitch100_fn
    totals["F1 %"] = 200 * totals.audio_pitch100_tp / denominator.replace(0, np.nan)
    display(totals.round(3))
display(comparison[comparison.rate == 0].groupby(["instrument", "setting"])[["audio_pitch100_fp"]].sum())
print("Next: independent chorales and a complete-pipeline check with repeat correction before changing production costs.")
